# Notebook 2 Bases de Datos Avanzadas - MongoDB

En este notebook ...

## 1. Librerías y dependencias

In [1]:
%pip install pandas
%pip install pymongo
%pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
from pymongo import MongoClient
import hashlib
import json
import os
from sentence_transformers import SentenceTransformer

## 2. Análisis exploratorio de datos

## 3. Conexión y creación de BD "Política"

In [8]:
try:
    client = MongoClient("mongodb://localhost:30001/?directConnection=true")

    # Creamos la BD Política
    db = client["Política"]
    coleccion = db["Discursos"]
    # Creamos coleccion "Discursos" e insertamos un dato de prueba
    coleccion.insert_one({"titulo": "Prueba", "autor": "Bruno"})

    print("Conexión exitosa! BD y colección creadas")
except Exception as e:
    print(f"Error de conexión: {e}")

Conexión exitosa! BD y colección creadas


> Conéctate a MongoDB desde la terminal
```bash
docker exec -it mongo-primary mongo --port 30001
```

> Ahora probar los comandos de MongoDB en la terminal.
```javascript
show dbs                        // ver todas las bases de datos
use Política                    // seleccionar la BD
show collections                // ver colecciones
db.Discursos.find()             // ver documentos
db.Discursos.find().pretty()    // ver documentos formateado
db.Discursos.deleteMany({})     // eliminar todos los elementos de la base de datos
db.Discursos.countDocuments({}) // contamos la cantidad de documentos dentro de la colección Discrusos

```

In [7]:
# Eliminamos el dato de prueba
coleccion.delete_one({"titulo": "Prueba", "autor": "Bruno"})

DeleteResult({'n': 1, 'electionId': ObjectId('7fffffff0000000000000003'), 'opTime': {'ts': Timestamp(1780107272, 1), 't': 3}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1780107272, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1780107272, 1)}, acknowledged=True)

> Volver a probar el comando para buscar discursos, verán que ya no hay

Testeamos que los 3 nodos se encuentren activos

In [5]:
primary    = MongoClient("mongodb://localhost:30001/?directConnection=true")
secondary1 = MongoClient("mongodb://localhost:30002/?directConnection=true")
secondary2 = MongoClient("mongodb://localhost:30003/?directConnection=true")

for nombre, cliente in [("Primary", primary), ("Secondary1", secondary1), ("Secondary2", secondary2)]:
    resultado = cliente.admin.command("ping")
    print(f"{'✅' if resultado['ok'] else '❌'} {nombre}")

✅ Primary
✅ Secondary1
✅ Secondary2


## 4. Inserción de documentos a Mongo


In [ ]:
# Guardamos la ruta de la carpeta que contiene los .txt
ruta_carpeta = "./DiscursosOriginales"

# Cargamos el modelo de Sentence-Transformers
modelo_transformer = SentenceTransformer('all-MiniLM-L6-v2')

print("==============  Inicio del procesamiento de discursos  ==============")

# Recorremos cada archivo de texto dentro de la carpeta DiscursosOriginales
for archivo in os.listdir(ruta_carpeta):
    if archivo.endswith(".txt"):
        ruta_completa = os.path.join(ruta_carpeta, archivo)

        try:
            with open(ruta_completa, 'r', encoding='utf-8') as texto:
                texto_discurso = texto.read()
            
            # Creamos el hash SHA-256 del discurso el cual será utilizado como id_unico en MongoDB
            hash_id_unico = hashlib.sha256(texto_discurso.encode('utf-8')).hexdigest()

            # Creamos el embedding del texto
            vector_embedding = modelo_transformer.encode(texto_discurso).tolist()

            # Armamos el JSON que se guardará en la BD
            documento_final = {
                "_id": hash_id_unico,
                "texto": texto_discurso,
                "embedding": vector_embedding
            }

            # Insertamos el nuevo JSON creado a la base de datos
            coleccion.insert_one(documento_final)
            print(f"  ✅ Procesado e insertado: {archivo}")

        except Exception as e:
            print(f"❌ Error al procesar el archivo {archivo}: {e}")

print("\n ==============  Termino del procesamiento de los discursos  ==============")
# Pruebas


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

==============  Inicio del procesamiento de discursos  ==============
  ✅ Procesado e insertado: 100020.txt
  ✅ Procesado e insertado: 100033.txt
  ✅ Procesado e insertado: 100172.txt
  ✅ Procesado e insertado: 100178.txt
  ✅ Procesado e insertado: 100227.txt
  ✅ Procesado e insertado: 100260.txt
  ✅ Procesado e insertado: 100296.txt
  ✅ Procesado e insertado: 100302.txt
  ✅ Procesado e insertado: 100385.txt
  ✅ Procesado e insertado: 100422.txt
  ✅ Procesado e insertado: 100454.txt
  ✅ Procesado e insertado: 100509.txt
  ✅ Procesado e insertado: 100547.txt
  ✅ Procesado e insertado: 100603.txt
  ✅ Procesado e insertado: 100990.txt
  ✅ Procesado e insertado: 101030.txt
  ✅ Procesado e insertado: 101041.txt
  ✅ Procesado e insertado: 101075.txt
  ✅ Procesado e insertado: 101103.txt
  ✅ Procesado e insertado: 101169.txt
  ✅ Procesado e insertado: 101255.txt
  ✅ Procesado e insertado: 101299.txt
  ✅ Procesado e insertado: 101310.txt
  ✅ Procesado e insertado: 101337.txt
  ✅ Procesado e in

## 5. Consulta textual y similitud coseno.

In [7]:
import numpy as np
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer


print("Cargando el modelo para las consultas...")
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def similitud_coseno(v1, v2):
    """Calcula la similitud coseno entre dos vectores usando NumPy."""
    arr1 = np.array(v1)
    arr2 = np.array(v2)
    
    num = np.dot(arr1, arr2)
    den = np.linalg.norm(arr1) * np.linalg.norm(arr2)
    
    if den == 0:
        return 0.0
    return float(num / den)

def buscar_top_5(consulta_texto):
    # 1. Generar embedding de la consulta
    embedding_consulta = model.encode(consulta_texto).tolist()
    
    # 2. Traer todos los documentos de la base de datos
    documentos = list(coleccion.find({}, {"_id": 1, "nombre_archivo": 1, "texto": 1, "embedding": 1}))
    
    if not documentos:
        print("La base de datos está vacía. Ejecuta primero el script de preprocesamiento.")
        return
    
    resultados = []
    
    # 3. Calcular la similitud para cada documento
    for doc in documentos:
        similitud = similitud_coseno(embedding_consulta, doc["embedding"])
        resultados.append({
            "_id": doc["_id"],
            "nombre_archivo": doc.get("nombre_archivo", "Desconocido"),
            "texto_corto": doc["texto"][:200] + "...", # Fragmento para visualización
            "similitud": similitud
        })
    
    # 4. Ordenar de mayor a menor según el puntaje de similitud
    resultados_ordenados = sorted(resultados, key=lambda x: x["similitud"], reverse=True)
    
    # 5. Mostrar los Top 5 resultados por consola
    print(f"\n=== RESULTADOS PARA LA CONSULTA: '{consulta_texto}' ===")
    for i, res in enumerate(resultados_ordenados[:5], start=1):
        print(f"\n[Top {i}] - Similitud: {res['similitud']:.4f}")
        print(f"ID (SHA-256): {res['_id']}")
        print(f"Archivo: {res['nombre_archivo']}")
        print(f"Extracto: {res['texto_corto']}")
        print("-" * 60)


Cargando el modelo para las consultas...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9872.92it/s]


In [8]:
buscar_top_5("Hola")


=== RESULTADOS PARA LA CONSULTA: 'Hola' ===

[Top 1] - Similitud: 0.5215
ID (SHA-256): 9926c7a47853800c3a5f07eb5cf808db898e3b886b284cefd9238f0b6b9b9eef
Archivo: 94030.txt
Extracto: Muy buenos días:

 

Gracias por venir a esta casa de todos, que es La Moneda, y gracias por comprometerse por una causa tan urgente y tan noble, como es proteger la sobrevivencia de la humanidad en e...
------------------------------------------------------------

[Top 2] - Similitud: 0.5205
ID (SHA-256): 41914b2000672baef74b7036ea6bd549082bee7071c34e3e8ed4707141305bae
Archivo: 79433.txt
Extracto: Muy buenos días:

 

En primer lugar, muchas gracias, porque pocas veces uno tiene el privilegio de poder participar en una inauguración tan querida, tan sentida y, al mismo tiempo, con esta maravillo...
------------------------------------------------------------

[Top 3] - Similitud: 0.5102
ID (SHA-256): c6633a0acfd06f0071e52f428279a5e860607529b508fd8550f115ac5d2e1572
Archivo: 99842.txt
Extracto: Muy buenos día